In [ ]:
import zipfile
import os

zip_path = "/content/house-prices-advanced-regression-techniques.zip"
extract_path = "/content/house_price_data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print(os.listdir(extract_path))

Dataset extracted successfully!
['sample_submission.csv', 'test.csv', 'train.csv', 'data_description.txt']


In [ ]:
import pandas as pd
import os

data_path = "/content/house_price_data"

train = pd.read_csv(os.path.join(data_path, "train.csv"))
test = pd.read_csv(os.path.join(data_path, "test.csv"))

print("Training dataset shape:", train.shape)
print("Testing dataset shape:", test.shape)

print("\nFirst 5 rows of training data:")
display(train.head())

Training dataset shape: (1460, 81)
Testing dataset shape: (1459, 80)

First 5 rows of training data:


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [ ]:
print("Dataset information:")
train.info()

print("\nMissing values in the top 15 columns:")
print(train.isnull().sum().sort_values(ascending=False).head(15))

print("\nHouse price statistics:")
print(train["SalePrice"].describe())

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   

In [ ]:
# Step 2: Prepare the data for machine learning

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
import numpy as np

train = pd.read_csv('/content/house_price_data/train.csv')

X = train.drop(columns=['SalePrice', 'Id'])
y = train['SalePrice']

numeric_cols = X.select_dtypes(include=['number']).columns
categorical_cols = X.select_dtypes(exclude=['number']).columns

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols)
])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Data preparation completed!")
print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))

Data preparation completed!
Training rows: 1168
Validation rows: 292


In [ ]:
# Step 3: Train and compare two machine-learning models

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

results = []

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_val)

    mae = mean_absolute_error(y_val, predictions)
    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    r2 = r2_score(y_val, predictions)

    results.append({
        'Model': name,
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'R2 Score': round(r2, 4)
    })

results_df = pd.DataFrame(results)
print("Model training and evaluation completed!")
display(results_df)

Model training and evaluation completed!


,Model,MAE,RMSE,R2 Score
0,Linear Regression,20466.13,31294.99,0.8723
1,Random Forest,17394.56,28396.30,0.8949


In [ ]:
# Step 4: Display final results neatly

print("HOUSE PRICE PREDICTION - FINAL RESULTS")
print("--------------------------------------")
print(results_df.to_string(index=False))

best_model = results_df.sort_values(
    by="R2 Score", ascending=False
).iloc[0]

print("\nBest model:", best_model["Model"])
print("Best R2 Score:", best_model["R2 Score"])
print("\nConclusion: Random Forest performed better")
print("than Linear Regression on the validation data.")

HOUSE PRICE PREDICTION - FINAL RESULTS
--------------------------------------
            Model      MAE     RMSE  R2 Score
Linear Regression 20466.13 31294.99    0.8723
    Random Forest 17394.56 28396.30    0.8949

Best model: Random Forest
Best R2 Score: 0.8949

Conclusion: Random Forest performed better
than Linear Regression on the validation data.


In [ ]:
print("""
CONCLUSION

This project developed a machine-learning system to predict
house prices using property features from the housing dataset.

Two regression algorithms were evaluated: Linear Regression
and Random Forest. Their performance was compared using
Mean Absolute Error (MAE), Root Mean Squared Error (RMSE),
and the R-squared (R2) score.

Linear Regression achieved an MAE of 20466.13, an RMSE of
31294.99, and an R2 score of 0.8723.

Random Forest achieved an MAE of 17394.56, an RMSE of
28396.30, and an R2 score of 0.8949.

Random Forest performed better on the validation dataset
because it had lower prediction errors and a higher R2 score.
This project demonstrates how machine learning can help
estimate house prices from property characteristics.

The model can be improved further through feature selection,
hyperparameter tuning, and testing on additional data.
""")


CONCLUSION

This project developed a machine-learning system to predict
house prices using property features from the housing dataset.

Two regression algorithms were evaluated: Linear Regression
and Random Forest. Their performance was compared using
Mean Absolute Error (MAE), Root Mean Squared Error (RMSE),
and the R-squared (R2) score.

Linear Regression achieved an MAE of 20466.13, an RMSE of
31294.99, and an R2 score of 0.8723.

Random Forest achieved an MAE of 17394.56, an RMSE of
28396.30, and an R2 score of 0.8949.

Random Forest performed better on the validation dataset
because it had lower prediction errors and a higher R2 score.
This project demonstrates how machine learning can help
estimate house prices from property characteristics.

The model can be improved further through feature selection,
hyperparameter tuning, and testing on additional data.



In [ ]:
# Load the second dataset
test_data = pd.read_csv('/content/house_price_data/test.csv')

# Prepare test data using the same features as training data
X_test = test_data.drop(columns=['Id'])

# Train the best model (Random Forest) on all training data
best_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

best_pipeline.fit(X, y)

# Predict house prices for the second dataset
test_predictions = best_pipeline.predict(X_test)

# Display predictions
predictions_df = pd.DataFrame({
    'Id': test_data['Id'],
    'Predicted Sale Price': test_predictions.round(2)
})

print("Predictions generated successfully!")
display(predictions_df.head(10))
print("Total predictions:", len(predictions_df))

Predictions generated successfully!


,Id,Predicted Sale Price
0,1461,129836.08
1,1462,156844.34
2,1463,182952.90
3,1464,182297.72
4,1465,195425.64
5,1466,185252.05
6,1467,169004.14
7,1468,176413.70
8,1469,182231.60
9,1470,120250.41


Total predictions: 1459


In [ ]:
predictions_df.to_csv(
    '/content/House_Price_Predictions.csv',
    index=False
)

print("Predictions saved successfully!")

Predictions saved successfully!


In [ ]:
from google.colab import files
files.download('/content/House_Price_Predictions.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>